# Essentials AI · Demo Day

**Objectif :** Créer et présenter une application d'IA.

Lance ce notebook depuis le dossier `notebooks/`.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath("../labs"))   # brewly, fakeapi
os.chdir("../data")                               # penguins.csv, titanic.csv, tips.csv

---
## Projet Demo Day · Un assistant de support intelligent

Ton projet de fin d'Essentials : un **assistant de support** pour CloudSync, qui combine ce que tu as appris. Il **route** la demande, **retrouve** les bons documents, **répond** avec ses sources, et il est **évalué**.

Les exercices ci-dessous construisent la version testable, avec un faux modèle. Tu remplaceras ensuite le faux modèle par le vrai (`await llm.complete(...)`) dans une cellule de la leçon RAG, puis tu en feras une démo avec n8n ou une petite interface.

L'architecture :

```markdown
question ──► routeur ──┬─► « humain » (résiliation, réclamation) ─► message de transfert
                       └─► recherche top-k ─► prompt avec sources ─► LLM ─► réponse + sources
```

**À faire :**

- [ ] Remplacer le faux modèle par le vrai LLM dans une cellule de la leçon RAG et comparer les réponses
- [ ] Construire la même chaîne dans n8n (Webhook → Code → modèle de langage → réponse)
- [ ] Préparer 5 diapositives de démo avec Gamma : problème, architecture, démo, évaluation, limites
- [ ] Publier le code et une vidéo de démo de 2 minutes sur GitHub

### Exercice 1 · Le pipeline de l'assistant

Écris `assistant(question, recherche, modele)`. Si la question contient `résilier` ou `réclamation` (sans tenir compte de la casse), renvoie `{"route": "humain", "reponse": "Je transmets ta demande à un conseiller.", "sources": []}`. Sinon : récupère les 2 meilleurs `id` avec `recherche(question, 2)`, construis un prompt qui contient la question et le texte de ces documents (cherche-les dans `DOCS`), appelle `modele(prompt)` et renvoie `{"route": "rag", "reponse": ..., "sources": [les 2 id]}`.

<details><summary>Indice</summary>

Commence par le test de routage (question.lower()). Pour le texte des documents, un dictionnaire id → texte est pratique.
</details>

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from cloudsync import DOCS

vect = TfidfVectorizer()
mat = vect.fit_transform([d["titre"] + ". " + d["texte"] for d in DOCS])

def recherche(question, k):
    s = cosine_similarity(vect.transform([question]), mat)[0]
    return [DOCS[i]["id"] for i in s.argsort()[::-1][:k]]

def modele_test(prompt):
    return "Réponse fondée sur " + str(prompt.count("[")) + " documents."

def assistant(question, recherche, modele):
    pass

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
vus = []
espion = lambda p: vus.append(p) or "ok"
r = assistant("Combien coûte l'offre Pro par mois ?", recherche, espion)
assert r["route"] == "rag" and r["reponse"] == "ok", f"Pour une question de prix, route rag attendue ; tu renvoies {r}."
assert r["sources"] == recherche("Combien coûte l'offre Pro par mois ?", 2), "sources doit contenir les 2 id renvoyés par la recherche."
assert "Combien coûte l'offre Pro" in vus[0] and "11,99" in vus[0], "Le prompt doit contenir la question et le texte des documents retrouvés."
h = assistant("Je souhaite faire une RÉCLAMATION", recherche, espion)
assert h == {"route": "humain", "reponse": "Je transmets ta demande à un conseiller.", "sources": []}, f"Les réclamations vont à un humain ; tu renvoies {h}."
assert len(vus) == 1, "Quand la demande part vers un humain, le modèle ne doit pas être appelé."
print("Bravo, tout est correct.")

### Exercice 2 · Le tableau de bord d'évaluation

En réutilisant ton `assistant` (recopie-le avec `recherche`), calcule sur `QUESTIONS_TEST` : `taux_sources`, la proportion de questions dont la réponse cite au moins un document pertinent (dans `sources`), et `taux_humain`, la proportion de questions envoyées à un humain.

<details><summary>Indice</summary>

Construis la liste des (question, résultat), puis deux moyennes de booléens.
</details>

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from cloudsync import DOCS, QUESTIONS_TEST

vect = TfidfVectorizer()
mat = vect.fit_transform([d["titre"] + ". " + d["texte"] for d in DOCS])

def recherche(question, k):
    s = cosine_similarity(vect.transform([question]), mat)[0]
    return [DOCS[i]["id"] for i in s.argsort()[::-1][:k]]

modele_test = lambda prompt: "ok"

# recopie ta fonction assistant ici

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
res = [(q, assistant(q["question"], recherche, lambda p: "ok")) for q in QUESTIONS_TEST]
exp_s = sum(any(p in r["sources"] for p in q["pertinents"]) for q, r in res) / len(res)
exp_h = sum(r["route"] == "humain" for _, r in res) / len(res)
assert abs(taux_sources - exp_s) < 1e-9, f"taux_sources vaut {taux_sources}, j'attendais {exp_s}."
assert abs(taux_humain - exp_h) < 1e-9 and exp_h == 1 / 8, f"taux_humain vaut {taux_humain} ; une question sur 8 parle de résiliation."
print("Bravo, tout est correct.")